# Evaluación de modelos en selección sobre árboles

Activa la GPU en *Entorno de ejecución > Cambiar tipo de entorno*. Luego ejecuta todo, o edita primero la celda de configuración. Todos los modelos reciben los mismos árboles.

In [ ]:
!pip install -q -U transformers accelerate mistral-common
!pip install flash-linear-attention causal-conv1d
!nvidia-smi -L

## Código del ambiente

In [ ]:
!mkdir -p seleccion_arbol

In [ ]:
%%writefile seleccion_arbol/verificador.py
def leer_datos():
    n, c = map(int, input().split())
    ganancias = list(map(int, input().split()))
    aristas = [[] for _ in range(n)]
    for _ in range(n - 1):
        a, b = map(int, input().split())
        aristas[a - 1].append(b - 1)
        aristas[b - 1].append(a - 1)
    return n, c, ganancias, aristas

def retornar_respuesta(costo):
    print(costo)


def calcular_optimo(n, c, ganancias, aristas):
    # Recorro el arbol para inventarme un orden cualquiera
    padre = [-10000] * n
    orden = [0]
    for nodo_padre in orden:
        for nodo_hijo in aristas[nodo_padre]:
            if nodo_hijo != padre[nodo_padre]:
                padre[nodo_hijo] = nodo_padre
                orden.append(nodo_hijo)

    no_elegir = [0] * n
    elegir = ganancias[:]
    for nodo_padre in reversed(orden[1:]):
        p = padre[nodo_padre]
        # Si no elijo ahora
        # ¿Me conviene haber elegido antes o no?
        no_elegir[p] += max(no_elegir[nodo_padre], elegir[nodo_padre])

        # Si se me ocurre elegir este
        # 1) No elijo el anterior -> Salgo gratis
        # 2) Si elijo el anterior -> Pago 2c
        elegir[p] += max(no_elegir[nodo_padre], elegir[nodo_padre] - 2 * c)

    return max(no_elegir[0], elegir[0])


def solve():
    n, c, ganancias, aristas = leer_datos()
    print(calcular_optimo(n, c, ganancias, aristas))


if __name__ == '__main__':
    solve()


In [ ]:
%%writefile seleccion_arbol/instancia.py
from dataclasses import dataclass

from verificador import calcular_optimo


@dataclass(frozen=True)
class Instancia:
    # Los nodos se numeran desde 1: valores[0] es el valor del nodo 1.
    valores: tuple[int, ...]
    aristas: tuple[tuple[int, int], ...]
    costo: int

    @property
    def cantidad_nodos(self) -> int:
        return len(self.valores)

    def vecinos(self) -> dict[int, list[int]]:
        vecinos = {nodo: [] for nodo in range(1, self.cantidad_nodos + 1)}
        for a, b in self.aristas:
            vecinos[a].append(b)
            vecinos[b].append(a)
        return vecinos

    def puntaje(self, seleccion: set[int]) -> int:
        # Cada nodo elegido aporta su valor, menos el costo por cada vecino también elegido.
        vecinos = self.vecinos()
        return sum(
            self.valores[nodo - 1] - self.costo * sum(vecino in seleccion for vecino in vecinos[nodo])
            for nodo in seleccion
        )

    def puntaje_voraz(self) -> int:
        # La regla simple: elegir todos los nodos positivos, o ninguno si eso resta puntos.
        positivos = {i + 1 for i, valor in enumerate(self.valores) if valor > 0}
        return max(0, self.puntaje(positivos))

    def puntaje_optimo(self) -> int:
        # El verificador numera los nodos desde 0.
        aristas = [[] for _ in self.valores]
        for a, b in self.aristas:
            aristas[a - 1].append(b - 1)
            aristas[b - 1].append(a - 1)
        return calcular_optimo(self.cantidad_nodos, self.costo, list(self.valores), aristas)





In [ ]:
%%writefile seleccion_arbol/generador.py
import random
from dataclasses import dataclass

from instancia import Instancia

INTENTOS_MAXIMOS = 2000


@dataclass(frozen=True)
class Nivel:
    nodos: int
    valor_minimo: int
    valor_maximo: int
    costo_maximo: int
    # Fracción máxima del óptimo que puede lograr la regla de elegir todos los positivos.
    voraz_maximo: float = 0.7

    def __post_init__(self):
        if self.nodos < 1:
            raise ValueError("El árbol debe tener al menos un nodo")
        if self.valor_minimo > self.valor_maximo:
            raise ValueError("Se requiere valor_minimo <= valor_maximo")
        if self.valor_maximo < 1:
            raise ValueError("Se requiere valor_maximo >= 1 para que exista un puntaje positivo")
        if self.costo_maximo < 1:
            raise ValueError("Se requiere costo_maximo >= 1")
        if not 0 <= self.voraz_maximo <= 1:
            raise ValueError("Se requiere 0 <= voraz_maximo <= 1")


# Currículum de menor a mayor dificultad.
NIVELES = [
    Nivel(nodos=3,  valor_minimo=1,   valor_maximo=10, costo_maximo=3),
    Nivel(nodos=4,  valor_minimo=1,   valor_maximo=10, costo_maximo=3),
    Nivel(nodos=5,  valor_minimo=-5,  valor_maximo=10, costo_maximo=3),
    Nivel(nodos=7,  valor_minimo=-5,  valor_maximo=10, costo_maximo=4),
    Nivel(nodos=10, valor_minimo=-10, valor_maximo=15, costo_maximo=5),
    Nivel(nodos=15, valor_minimo=-10, valor_maximo=20, costo_maximo=6),
    Nivel(nodos=25, valor_minimo=-20, valor_maximo=30, costo_maximo=8),
    Nivel(nodos=40, valor_minimo=-30, valor_maximo=40, costo_maximo=10),
]


def _crear_instancia(nivel: Nivel, rng: random.Random) -> Instancia:
    # Cada nodo se une a uno anterior, y luego se barajan los números para ocultar la raíz.
    numeros = list(range(1, nivel.nodos + 1))
    rng.shuffle(numeros)
    aristas = tuple(
        (numeros[i], numeros[rng.randrange(i)])
        for i in range(1, nivel.nodos)
    )
    valores = tuple(rng.randint(nivel.valor_minimo, nivel.valor_maximo) for _ in numeros)
    return Instancia(valores, aristas, rng.randint(1, nivel.costo_maximo))


def generar_instancia(nivel: Nivel, rng: random.Random | None = None) -> Instancia:
    rng = rng or random.Random()

    for _ in range(INTENTOS_MAXIMOS):
        instancia = _crear_instancia(nivel, rng)
        optimo = instancia.puntaje_optimo()

        if optimo > 0 and instancia.puntaje_voraz() <= nivel.voraz_maximo * optimo:
            return instancia

    raise RuntimeError(f"No se encontró una instancia para {nivel} en {INTENTOS_MAXIMOS} intentos")


In [ ]:
%%writefile seleccion_arbol/prompt.py
from instancia import Instancia


def generar_prompt(instancia: Instancia) -> str:
    valores = "\n".join(f"{i}: {valor}" for i, valor in enumerate(instancia.valores, 1))
    aristas = "\n".join(f"{a}-{b}" for a, b in instancia.aristas) or "(ninguna)"



    return f"""Elige un conjunto de nodos de este árbol para obtener el mayor puntaje posible.

Valor de cada nodo:
{valores}

Conexiones:
{aristas}

Costo c = {instancia.costo}

Puntaje: cada nodo elegido aporta su valor menos c por cada vecino que también esté elegido. Los nodos no elegidos no aportan nada. Si no eliges ninguno, el puntaje es 0.

Termina con los números de los nodos elegidos, por ejemplo:
<respuesta>2 5 9</respuesta>
Si no eliges ninguno, escribe <respuesta></respuesta>."""


In [ ]:
%%writefile seleccion_arbol/modelo.py
import json
import urllib.error
import urllib.request
from abc import ABC, abstractmethod


class ModeloLLM(ABC):
    @abstractmethod
    def generar(self, prompt: str) -> str:
        pass


class ModeloLMStudio(ModeloLLM):
    def __init__(
        self,
        nombre: str = "qwen3.5-2b",
        url_base: str = "http://localhost:1235/v1",
        temperatura: float = 0.7,
        tokens_maximos: int = 12000,
        segundos_limite: float = 600,
    ):
        self.nombre = nombre
        self.url_base = url_base.rstrip("/")
        self.temperatura = temperatura
        self.tokens_maximos = tokens_maximos
        self.segundos_limite = segundos_limite

    def generar(self, prompt: str) -> str:
        cuerpo = {
            "model": self.nombre,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": self.temperatura,
            # Incluye el razonamiento oculto. Si se agota, el contenido puede llegar vacío.
            "max_tokens": self.tokens_maximos,
        }
        url = f"{self.url_base}/chat/completions"
        solicitud = urllib.request.Request(
            url,
            data=json.dumps(cuerpo).encode("utf-8"),
            headers={"Content-Type": "application/json"},
        )

        try:
            with urllib.request.urlopen(solicitud, timeout=self.segundos_limite) as respuesta:
                datos = json.load(respuesta)
        except urllib.error.HTTPError as error:
            detalle = error.read().decode("utf-8", errors="replace")
            raise RuntimeError(f"{url} respondió {error.code}: {detalle}") from error
        except (OSError, json.JSONDecodeError) as error:
            raise RuntimeError(f"No se pudo consultar {url}: {error}") from error

        try:
            return datos["choices"][0]["message"]["content"] or ""
        except (KeyError, IndexError, TypeError) as error:
            raise RuntimeError(f"Respuesta inesperada de {url}: {datos}") from error


In [ ]:
%%writefile seleccion_arbol/modelo_hf.py
import gc

from modelo import ModeloLLM

CIERRE_RAZONAMIENTO = "\n</think>\n\n"


def razonamiento_abierto(prompt: str, texto: str) -> bool:
    return "</think>" not in texto and (prompt.rstrip().endswith("<think>") or "<think>" in texto)


def extraer_respuesta(prompt: str, texto: str) -> str:
    # Devuelve lo escrito después del razonamiento. Si este quedó sin cerrar, no hay respuesta.
    if "</think>" in texto:
        return texto.rsplit("</think>", 1)[1].strip()
    if razonamiento_abierto(prompt, texto):
        return ""
    return texto.strip()


class ModeloTransformers(ModeloLLM):
    # El modelo se carga en la primera consulta: si ya está evaluado, no ocupa la GPU.
    def __init__(
        self,
        nombre: str,
        razonar: bool | None = None,
        temperatura: float = 0.7,
        tokens_maximos: int = 12000,
        tokens_razonamiento: int = 8192,
    ):
        self.nombre = nombre
        self.razonar = razonar  # None usa lo que el modelo haga por defecto
        self.temperatura = temperatura
        self.tokens_maximos = tokens_maximos
        self.tokens_razonamiento = tokens_razonamiento  # Al llegar aquí, se cierra el razonamiento abierto
        self.ultimo_tokens = None
        self.ultimo_texto = None
        self.ultimo_razonamiento_cerrado = None
        self._modelo = None
        self._tokenizador = None

    def _cargar(self):
        from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoTokenizer

        self._tokenizador = AutoTokenizer.from_pretrained(self.nombre)
        try:
            self._modelo = AutoModelForImageTextToText.from_pretrained(
                self.nombre, torch_dtype="auto", device_map="auto"
            )
        except ValueError:
            self._modelo = AutoModelForCausalLM.from_pretrained(
                self.nombre, torch_dtype="auto", device_map="auto"
            )

    def generar(self, prompt: str) -> str:
        import torch

        if self._modelo is None:
            self._cargar()

        opciones = {} if self.razonar is None else {"enable_thinking": self.razonar}
        texto_prompt = self._tokenizador.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
            **opciones,
        )
        entrada = self._tokenizador(texto_prompt, return_tensors="pt").to(self._modelo.device)
        largo_prompt = entrada["input_ids"].shape[1]
        limite = min(self.tokens_razonamiento, self.tokens_maximos)
        self.ultimo_razonamiento_cerrado = False

        with torch.inference_mode():
            salida = self._continuar(entrada, limite)

            # Terminar antes del límite significa que el modelo ya dejó de generar por sí mismo.
            if salida.shape[1] - largo_prompt == limite < self.tokens_maximos:
                if razonamiento_abierto(texto_prompt, self._decodificar(salida, largo_prompt)):
                    cierre = self._tokenizador(
                        CIERRE_RAZONAMIENTO, add_special_tokens=False, return_tensors="pt"
                    )["input_ids"].to(salida.device)
                    salida = torch.cat([salida, cierre], dim=1)
                    self.ultimo_razonamiento_cerrado = True
                restantes = self.tokens_maximos - (salida.shape[1] - largo_prompt)
                if restantes > 0:
                    salida = self._continuar(
                        {"input_ids": salida, "attention_mask": torch.ones_like(salida)}, restantes
                    )

        self.ultimo_tokens = salida.shape[1] - largo_prompt
        self.ultimo_texto = self._decodificar(salida, largo_prompt)
        return extraer_respuesta(texto_prompt, self.ultimo_texto)

    def _continuar(self, entrada, tokens):
        return self._modelo.generate(
            **entrada,
            max_new_tokens=tokens,
            do_sample=self.temperatura > 0,
            temperature=self.temperatura if self.temperatura > 0 else None,
        )

    def _decodificar(self, salida, largo_prompt: int) -> str:
        return self._tokenizador.decode(salida[0][largo_prompt:], skip_special_tokens=True)

    def liberar(self):
        import torch

        self._modelo = None
        self._tokenizador = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
%%writefile seleccion_arbol/entorno.py
import random
import re
from dataclasses import dataclass

from generador import Nivel, generar_instancia
from instancia import Instancia
from modelo import ModeloLLM, ModeloLMStudio
from prompt import generar_prompt

RECOMPENSA_FORMATO_INVALIDO = -1.0
RECOMPENSA_VORAZ = 0.5  # Igualar la regla simple vale esto; el resto se gana al superarla

PATRON_BLOQUE = re.compile(r"<respuesta>(.*?)</respuesta>", re.DOTALL)
PATRON_NODOS = re.compile(r"\s*(?:\d{1,6}(?:[\s,]+\d{1,6})*)?\s*", re.ASCII)


class ErrorFormato(ValueError):
    pass


@dataclass(frozen=True)
class Resultado:
    recompensa: float
    puntaje: int | None
    puntaje_voraz: int
    puntaje_optimo: int
    error: str | None


def parsear_respuesta(instancia: Instancia, respuesta: str) -> set[int]:
    # Se usa el último bloque porque el modelo puede razonar antes de responder.
    bloques = PATRON_BLOQUE.findall(respuesta)
    if not bloques:
        raise ErrorFormato("Falta el bloque <respuesta>...</respuesta>")
    if PATRON_NODOS.fullmatch(bloques[-1]) is None:
        raise ErrorFormato("El bloque <respuesta> solo debe contener números de nodos")

    nodos = [int(nodo) for nodo in re.split(r"[\s,]+", bloques[-1].strip()) if nodo]
    if len(nodos) != len(set(nodos)):
        raise ErrorFormato("Hay nodos repetidos")
    if any(not 1 <= nodo <= instancia.cantidad_nodos for nodo in nodos):
        raise ErrorFormato(f"Los nodos deben estar entre 1 y {instancia.cantidad_nodos}")
    return set(nodos)


def calcular_recompensa(puntaje: int, voraz: int, optimo: int) -> float:
    # Peor que la regla simple: de 0 a RECOMPENSA_VORAZ. Mejor que ella: de RECOMPENSA_VORAZ a 1.
    if optimo == voraz:
        # No hay nada que superar. El generador lo evita, pero una instancia a mano sí podría.
        return max(0, puntaje) / optimo if optimo > 0 else float(puntaje == 0)
    if puntaje <= voraz:
        return RECOMPENSA_VORAZ * max(0, puntaje) / voraz if voraz > 0 else 0.0
    return RECOMPENSA_VORAZ + (1 - RECOMPENSA_VORAZ) * (puntaje - voraz) / (optimo - voraz)


def evaluar_respuesta(instancia: Instancia, respuesta: str) -> Resultado:
    voraz = instancia.puntaje_voraz()
    optimo = instancia.puntaje_optimo()

    try:
        seleccion = parsear_respuesta(instancia, respuesta)
    except ErrorFormato as error:
        return Resultado(RECOMPENSA_FORMATO_INVALIDO, None, voraz, optimo, str(error))

    puntaje = instancia.puntaje(seleccion)
    return Resultado(calcular_recompensa(puntaje, voraz, optimo), puntaje, voraz, optimo, None)


class EntornoSeleccion:
    # Cada episodio tiene un solo paso: el modelo entrega la selección completa.
    def __init__(self, semilla: int | None = None):
        self._rng = random.Random(semilla)
        self.instancia: Instancia | None = None

    def reset(self, nivel: Nivel) -> str:
        self.instancia = generar_instancia(nivel, self._rng)
        return generar_prompt(self.instancia)

    def step(self, respuesta: str) -> Resultado:
        if self.instancia is None:
            raise RuntimeError("Hay que llamar a reset antes de step")

        instancia, self.instancia = self.instancia, None
        return evaluar_respuesta(instancia, respuesta)


def jugar_episodio(entorno: EntornoSeleccion, modelo: ModeloLLM, nivel: Nivel) -> Resultado:
    prompt = entorno.reset(nivel)
    return entorno.step(modelo.generar(prompt))


if __name__ == "__main__":
    from generador import NIVELES

    print(jugar_episodio(EntornoSeleccion(), ModeloLMStudio(), NIVELES[0]))


In [ ]:
%%writefile seleccion_arbol/evaluacion.py
import json
import random
import time
from pathlib import Path

from entorno import evaluar_respuesta
from generador import NIVELES, generar_instancia
from instancia import Instancia
from modelo import ModeloLLM
from prompt import generar_prompt


def instancia_del_episodio(nivel: int, episodio: int, semilla: int = 0) -> Instancia:
    # Depende solo de estos datos, así todos los modelos reciben los mismos árboles.
    return generar_instancia(NIVELES[nivel - 1], random.Random(f"{semilla}-{nivel}-{episodio}"))


def leer_resultados(ruta) -> list[dict]:
    ruta = Path(ruta)
    if not ruta.exists():
        return []
    return [json.loads(linea) for linea in ruta.read_text(encoding="utf-8").splitlines() if linea.strip()]


def evaluar_modelo(
    modelo: ModeloLLM,
    nombre: str,
    niveles: list[int],
    episodios: int,
    ruta,
    semilla: int = 0,
) -> None:
    # Cada episodio se guarda al terminar, y al repetir la llamada se saltan los ya guardados.
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    hechos = {
        (fila["modelo"], fila["nivel"], fila["episodio"], fila["semilla"])
        for fila in leer_resultados(ruta)
    }

    for nivel in niveles:
        for episodio in range(1, episodios + 1):
            if (nombre, nivel, episodio, semilla) in hechos:
                continue

            instancia = instancia_del_episodio(nivel, episodio, semilla)
            inicio = time.perf_counter()
            try:
                respuesta = modelo.generar(generar_prompt(instancia))
            except RuntimeError as error:
                # Es una falla de infraestructura: no se guarda, para reintentarla al reanudar.
                print(f"{nombre} nivel {nivel} ep {episodio}: falló la consulta: {error}")
                continue

            resultado = evaluar_respuesta(instancia, respuesta)
            fila = {
                "modelo": nombre,
                "nivel": nivel,
                "episodio": episodio,
                "semilla": semilla,
                "nodos": instancia.cantidad_nodos,
                "recompensa": resultado.recompensa,
                "puntaje": resultado.puntaje,
                "puntaje_voraz": resultado.puntaje_voraz,
                "puntaje_optimo": resultado.puntaje_optimo,
                "error": resultado.error,
                "segundos": round(time.perf_counter() - inicio, 1),
                "tokens": getattr(modelo, "ultimo_tokens", None),
                "respuesta": respuesta,
                "texto_completo": getattr(modelo, "ultimo_texto", None),
                "razonamiento_cerrado": getattr(modelo, "ultimo_razonamiento_cerrado", None),
            }
            with ruta.open("a", encoding="utf-8") as archivo:
                archivo.write(json.dumps(fila, ensure_ascii=False) + "\n")
            print(
                f"{nombre} nivel {nivel} ep {episodio}: recompensa={resultado.recompensa:.3f} "
                f"puntaje={resultado.puntaje} óptimo={resultado.puntaje_optimo} "
                f"{fila['segundos']}s {fila['tokens']}tok"
            )


def resumir(ruta):
    import pandas as pd

    tabla = pd.DataFrame(leer_resultados(ruta))
    if tabla.empty:
        return tabla
    tabla["optimo"] = tabla["recompensa"] == 1.0
    tabla["invalido"] = tabla["recompensa"] == -1.0
    tabla["supera_voraz"] = tabla["puntaje"].notna() & (tabla["puntaje"] > tabla["puntaje_voraz"])
    return tabla.groupby(["modelo", "nivel"]).agg(
        episodios=("recompensa", "size"),
        recompensa=("recompensa", "mean"),
        optimos=("optimo", "mean"),
        supera_voraz=("supera_voraz", "mean"),
        invalidos=("invalido", "mean"),
        segundos=("segundos", "mean"),
        tokens=("tokens", "mean"),
    ).round(3)


In [ ]:
%%writefile seleccion_arbol/test_ambiente.py
import random
import tempfile
import unittest
from itertools import combinations
from pathlib import Path

from entorno import EntornoSeleccion, calcular_recompensa, evaluar_respuesta
from evaluacion import evaluar_modelo, leer_resultados, resumir
from generador import NIVELES, Nivel, generar_instancia
from modelo_hf import extraer_respuesta
from instancia import Instancia
from modelo import ModeloLLM
from prompt import generar_prompt


def fuerza_bruta(instancia: Instancia) -> int:
    nodos = range(1, instancia.cantidad_nodos + 1)
    return max(
        instancia.puntaje(set(seleccion))
        for tamano in range(instancia.cantidad_nodos + 1)
        for seleccion in combinations(nodos, tamano)
    )


def mejor_seleccion(instancia: Instancia) -> set[int]:
    nodos = range(1, instancia.cantidad_nodos + 1)
    return max(
        (set(s) for t in range(instancia.cantidad_nodos + 1) for s in combinations(nodos, t)),
        key=instancia.puntaje,
    )


# Camino 1-2-3 con c = 2: elegir 1 y 3 da 10; elegir los tres da 16 - 2*2*2 = 8.
CAMINO = Instancia((5, 6, 5), ((1, 2), (2, 3)), 2)


class PruebasPuntaje(unittest.TestCase):
    def test_puntaje_a_mano(self):
        self.assertEqual(CAMINO.puntaje(set()), 0)
        self.assertEqual(CAMINO.puntaje({2}), 6)
        self.assertEqual(CAMINO.puntaje({1, 3}), 10)
        self.assertEqual(CAMINO.puntaje({1, 2}), 11 - 2 * 2)
        self.assertEqual(CAMINO.puntaje({1, 2, 3}), 8)
        self.assertEqual(CAMINO.puntaje_optimo(), 10)

    def test_valores_negativos_y_un_nodo(self):
        self.assertEqual(Instancia((-3,), (), 1).puntaje_optimo(), 0)
        self.assertEqual(Instancia((4,), (), 1).puntaje_optimo(), 4)
        self.assertEqual(Instancia((-1, -2), ((1, 2),), 1).puntaje_optimo(), 0)

    def test_optimo_contra_fuerza_bruta(self):
        rng = random.Random(1)
        for _ in range(1500):
            nivel = Nivel(rng.randint(1, 11), -15, 20, 12, voraz_maximo=1.0)
            instancia = generar_instancia(nivel, rng)
            self.assertEqual(instancia.puntaje_optimo(), fuerza_bruta(instancia), instancia)


class PruebasGenerador(unittest.TestCase):
    def test_niveles(self):
        rng = random.Random(2)
        for nivel in NIVELES:
            for _ in range(30):
                instancia = generar_instancia(nivel, rng)
                optimo = instancia.puntaje_optimo()
                self.assertEqual(instancia.cantidad_nodos, nivel.nodos)
                self.assertEqual(len(instancia.aristas), nivel.nodos - 1)
                self.assertGreater(optimo, 0)
                # Un árbol es conexo: todos los nodos se alcanzan desde el 1.
                vecinos, vistos, pendientes = instancia.vecinos(), {1}, [1]
                while pendientes:
                    for vecino in vecinos[pendientes.pop()]:
                        if vecino not in vistos:
                            vistos.add(vecino)
                            pendientes.append(vecino)
                self.assertEqual(len(vistos), nivel.nodos)
                self.assertLessEqual(instancia.puntaje_voraz(), 0.7 * optimo)

    def test_reproducible(self):
        self.assertEqual(generar_instancia(NIVELES[4], random.Random(9)), generar_instancia(NIVELES[4], random.Random(9)))

    def test_parametros_invalidos(self):
        for argumentos in [(0, 1, 2, 1), (3, 5, 1, 1), (3, -5, 0, 1), (3, 1, 5, 0), (3, 1, 5, 1, 1.5), (3, 1, 5, 1, -0.1)]:
            with self.assertRaises(ValueError):
                Nivel(*argumentos)


class PruebasRecompensa(unittest.TestCase):
    def test_recompensas(self):
        # En CAMINO la regla simple logra 8 y el óptimo es 10.
        casos = [
            ("<respuesta>1 3</respuesta>", 1.0, 10),
            ("<respuesta>3, 1</respuesta>", 1.0, 10),
            ("Pienso...\n<respuesta>2</respuesta>\nMejor:\n<respuesta>1 3</respuesta>", 1.0, 10),
            ("<respuesta>2</respuesta>", 0.375, 6),
            ("<respuesta>1 2 3</respuesta>", 0.5, 8),
            ("<respuesta></respuesta>", 0.0, 0),
        ]
        for texto, recompensa, puntaje in casos:
            resultado = evaluar_respuesta(CAMINO, texto)
            self.assertAlmostEqual(resultado.recompensa, recompensa, msg=texto)
            self.assertEqual(resultado.puntaje, puntaje, texto)
            self.assertEqual(resultado.puntaje_voraz, 8)

    def test_tramos(self):
        # Los casos de la tabla: óptimo 20 con regla simple en 18, y óptimo 14 con regla simple en 3.
        for puntaje, voraz, optimo, esperado in [
            (18, 18, 20, 0.5), (19, 18, 20, 0.75), (20, 18, 20, 1.0),
            (2, 3, 14, 1 / 3), (3, 3, 14, 0.5), (7, 3, 14, 0.5 + 0.5 * 4 / 11), (14, 3, 14, 1.0),
            (-4, 3, 14, 0.0), (0, 3, 14, 0.0),
            (5, 0, 6, 0.5 + 0.5 * 5 / 6), (0, 0, 6, 0.0), (-2, 0, 6, 0.0),
            (14, 14, 14, 1.0), (7, 14, 14, 0.5),
        ]:
            self.assertAlmostEqual(calcular_recompensa(puntaje, voraz, optimo), esperado, msg=(puntaje, voraz, optimo))

    def test_recompensa_crece_con_el_puntaje_y_queda_en_rango(self):
        rng = random.Random(6)
        for _ in range(200):
            instancia = generar_instancia(NIVELES[3], rng)
            voraz, optimo = instancia.puntaje_voraz(), instancia.puntaje_optimo()
            recompensas = [calcular_recompensa(p, voraz, optimo) for p in range(-30, optimo + 1)]
            self.assertEqual(recompensas, sorted(recompensas))
            self.assertTrue(all(0 <= r <= 1 for r in recompensas))
            self.assertEqual(recompensas[-1], 1.0)
            self.assertEqual(calcular_recompensa(voraz, voraz, optimo), 0.5 if voraz > 0 else 0.0)

    def test_puntaje_negativo_no_baja_de_cero(self):
        instancia = Instancia((1, 1, 5), ((1, 2), (2, 3)), 3)
        resultado = evaluar_respuesta(instancia, "<respuesta>1 2</respuesta>")
        self.assertEqual(resultado.puntaje, -4)
        self.assertEqual(resultado.puntaje_voraz, 0)
        self.assertEqual(resultado.recompensa, 0.0)

    def test_optimo_cero(self):
        instancia = Instancia((-3,), (), 1)
        self.assertEqual(evaluar_respuesta(instancia, "<respuesta></respuesta>").recompensa, 1.0)
        self.assertEqual(evaluar_respuesta(instancia, "<respuesta>1</respuesta>").recompensa, 0.0)

    def test_formato_invalido(self):
        for texto in ["", "1 3", "<respuesta>1 3", "<respuesta>uno</respuesta>", "<respuesta>1 1</respuesta>",
                      "<respuesta>0</respuesta>", "<respuesta>4</respuesta>", "<respuesta>-1</respuesta>",
                      "<respuesta>٣</respuesta>", "<respuesta>" + "9" * 5000 + "</respuesta>"]:
            resultado = evaluar_respuesta(CAMINO, texto)
            self.assertEqual(resultado.recompensa, -1.0, texto[:30])
            self.assertIsNotNone(resultado.error)

    def test_textos_aleatorios_no_rompen(self):
        rng = random.Random(3)
        letras = "<>respuesta/0123456789 ,\n-٣"
        for _ in range(3000):
            texto = "".join(rng.choice(letras) for _ in range(rng.randint(0, 40)))
            if rng.random() < 0.5:
                texto = f"<respuesta>{texto}</respuesta>"
            self.assertTrue(-1.0 <= evaluar_respuesta(CAMINO, texto).recompensa <= 1.0)


class ModeloFuerzaBruta(ModeloLLM):
    def __init__(self, entorno):
        self.entorno = entorno

    def generar(self, prompt):
        assert prompt == generar_prompt(self.entorno.instancia)
        seleccion = " ".join(map(str, sorted(mejor_seleccion(self.entorno.instancia))))
        return f"Razonamiento.\n<respuesta>{seleccion}</respuesta>"


class PruebasEntorno(unittest.TestCase):
    def test_episodios_optimos(self):
        entorno = EntornoSeleccion(semilla=4)
        modelo = ModeloFuerzaBruta(entorno)
        for nivel in NIVELES[:5]:
            prompt = entorno.reset(nivel)
            resultado = entorno.step(modelo.generar(prompt))
            self.assertEqual(resultado.recompensa, 1.0)

    def test_step_requiere_reset(self):
        entorno = EntornoSeleccion(semilla=5)
        with self.assertRaises(RuntimeError):
            entorno.step("<respuesta></respuesta>")
        entorno.reset(NIVELES[0])
        entorno.step("<respuesta></respuesta>")
        with self.assertRaises(RuntimeError):
            entorno.step("<respuesta></respuesta>")


class ModeloFijo(ModeloLLM):
    def __init__(self, falla_en=None):
        self.prompts = []
        self.falla_en = falla_en

    def generar(self, prompt):
        self.prompts.append(prompt)
        if len(self.prompts) == self.falla_en:
            raise RuntimeError("sin memoria")
        return "<respuesta></respuesta>"


class PruebasEvaluacion(unittest.TestCase):
    def test_mismos_arboles_y_reanuda(self):
        with tempfile.TemporaryDirectory() as carpeta:
            ruta = Path(carpeta) / "sub" / "r.jsonl"
            a, b = ModeloFijo(falla_en=2), ModeloFijo()
            evaluar_modelo(a, "a", [2, 3], 2, ruta)
            self.assertEqual(len(leer_resultados(ruta)), 3)  # La falla no se guarda
            evaluar_modelo(a, "a", [2, 3], 2, ruta)
            self.assertEqual(len(leer_resultados(ruta)), 4)  # Solo reintenta la pendiente
            self.assertEqual(len(a.prompts), 5)
            evaluar_modelo(b, "b", [2, 3], 2, ruta)
            self.assertEqual(sorted(set(a.prompts)), sorted(b.prompts))
            resumen = resumir(ruta)
            self.assertEqual(resumen.loc[("b", 2), "episodios"], 2)
            self.assertEqual(resumen.loc[("b", 2), "recompensa"], 0.0)

    def test_extraer_respuesta(self):
        self.assertEqual(extraer_respuesta("p", "<respuesta>1</respuesta>"), "<respuesta>1</respuesta>")
        self.assertEqual(extraer_respuesta("p<think>\n", "duda\n</think>\n<respuesta>1</respuesta>"), "<respuesta>1</respuesta>")
        self.assertEqual(extraer_respuesta("p<think>\n", "<respuesta>1</respuesta> sigo pensando"), "")
        self.assertEqual(extraer_respuesta("p", "<think>a <respuesta>1</respuesta>"), "")


if __name__ == "__main__":
    unittest.main()


In [ ]:
import sys
sys.path.insert(0, 'seleccion_arbol')
!cd seleccion_arbol && python -m unittest test_ambiente 2>&1 | tail -3

## Configuración

Cada entrada se evalúa por separado. `razonar` activa o desactiva el razonamiento en los modelos que lo permiten. Si a los 8192 tokens el razonamiento sigue abierto, se cierra y el modelo continúa hasta `tokens_maximos`. Los resultados van a Drive, así que al repetir la evaluación tras un corte de sesión se continúa donde quedó.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUTA = '/content/drive/MyDrive/seleccion_arbol/resultados_con_tope.jsonl'
NIVELES_A_EVALUAR = [2, 3, 4, 5]
EPISODIOS = 5
SEMILLA = 0

MODELOS = {
    'Qwen3.5 2B': dict(nombre='Qwen/Qwen3.5-2B', razonar=True, tokens_maximos=12000),
    'Llama 3.2 1B': dict(nombre='unsloth/Llama-3.2-1B-Instruct', razonar=True, tokens_maximos=12000),
    'OLMo 2 1B': dict(nombre='allenai/OLMo-2-0425-1B-Instruct', razonar=True, tokens_maximos=12000),
    'DeepSeek R1 Distill 1.5B': dict(nombre='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', razonar=True, tokens_maximos=12000),
    'Ministral 3 3B Reasoning': dict(nombre='mistralai/Ministral-3-3B-Reasoning-2512', razonar=True, tokens_maximos=12000),
}

## Evaluación

In [ ]:
from evaluacion import evaluar_modelo, resumir
from modelo_hf import ModeloTransformers

for nombre, opciones in MODELOS.items():
    modelo = ModeloTransformers(**opciones)
    try:
        evaluar_modelo(modelo, nombre, NIVELES_A_EVALUAR, EPISODIOS, RUTA, SEMILLA)
    except Exception as error:
        # Un modelo que no carga no debe impedir evaluar los demás.
        print(f'{nombre}: se omite por {type(error).__name__}: {error}')
    finally:
        modelo.liberar()

## Resultados

Igualar la regla simple vale 0,5 de recompensa; una respuesta inválida vale -1.

In [ ]:
resumen = resumir(RUTA)
resumen

In [ ]:
ax = resumen['recompensa'].unstack('modelo').plot(marker='o', ylim=(-1.05, 1.05), ylabel='Recompensa media')
ax.axhline(0.5, color='gray', linestyle='--', label='Regla simple')
ax.legend();